<p style ="text-align:center">
    <img src="https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/BannerUFPR.png" width="700" alt="PPGOLD/PPGMNE Python:INTRO"  />
</p>


# Liga Data Science
## Prof. Eduardo Pécora

# Análise de Componentes Principais (PCA) · 3. PCA em modelos preditivos
Tempo estimado: **25** minutos

-----

## Objetivos

Após completar esta aula, você será capaz de:

* Reconhecer o vazamento de dados ao aplicar a PCA antes de separar treino e teste
* Montar um Pipeline (padronizar → PCA → classificador) com validação cruzada
* Escolher o número de componentes testando o desempenho
* Avaliar com honestidade quando a PCA compensa em um modelo

Nos notebooks anteriores usamos a PCA para **entender e resumir** dados (`PPGOLD_DS_9_1_PCA_Intuicao` e `PPGOLD_DS_9_2_PCA_Atlas`). Aqui a pergunta é outra:

> Usar PCA antes de um classificador **melhora** o modelo? E como fazer isso **sem trapacear**?

Vamos usar o conjunto *Breast Cancer Wisconsin*: **569 tumores**, **30 medidas** dos núcleos celulares e o diagnóstico (**maligno** ou **benigno**).

| Etapa | Pergunta |
|---|---|
| **A** | Por que 30 variáveis são "demais"? |
| **B** | Qual é o erro de aplicar a PCA antes de separar treino e teste? |
| **C** | Como montar o `Pipeline` correto? |
| **D** | Quantas componentes usar e se a PCA compensa |
| **E** | Avaliação final no conjunto de teste |

#### Bibliotecas e semente aleatória

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEMENTE = 31415
np.random.seed(SEMENTE)
sns.set_theme(style="whitegrid")

#### Carregando os dados

In [ ]:
cancer = load_breast_cancer()
X = pd.DataFrame(cancer["data"], columns=cancer["feature_names"])   # variáveis independentes
y = pd.Series(cancer["target"], name="diagnostico")                # 0 = maligno, 1 = benigno

print(X.shape)
print(y.value_counts().rename({0: "maligno", 1: "benigno"}))
X.head()

---
# A — Por que 30 variáveis são "demais"?

As 30 colunas são 10 medidas (raio, textura, perímetro, área…) calculadas de 3 formas (média, erro padrão e "pior" valor). Raio, perímetro e área de uma célula carregam quase a mesma informação.

In [ ]:
corr = X.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, xticklabels=False)
plt.title("Correlação entre as 30 variáveis")
plt.show()

### Tarefa 1 — Medindo a redundância
Conte quantos **pares de variáveis diferentes** têm correlação, em módulo, **maior que 0,9**.

Dica: a matriz é simétrica e a diagonal vale 1. Use só o triângulo superior: `np.triu(np.ones(corr.shape), k=1).astype(bool)`.

In [ ]:
# Tarefa 1 — seu código aqui

**Sua resposta:** _escreva aqui_

---
# B — O erro mais comum: vazamento de dados

Um roteiro que aparece em muitos tutoriais (inclusive na versão anterior desta aula):

```python
scaled = StandardScaler().fit_transform(X)        # 1. padroniza com TODOS os dados
x_pca  = PCA(n_components=7).fit_transform(scaled) # 2. PCA com TODOS os dados
X_train, X_test, ... = train_test_split(x_pca, y)  # 3. só agora separa treino e teste
```

### Tarefa 2 — Onde está o problema?
Leia as três linhas acima. Que informação do **conjunto de teste** "vaza" para o modelo antes da avaliação? Por que isso é um problema, mesmo que a PCA não use o diagnóstico `y`?

**Sua resposta:** _escreva aqui_

---
# C — O jeito certo: separar primeiro, e usar `Pipeline`

1. Separamos um **conjunto de teste** que só será usado no final (Etapa E).
2. No treino, usamos **validação cruzada**: o treino é dividido em 5 partes, e cada parte serve de validação uma vez.
3. Um `Pipeline` encadeia **padronizar → PCA → classificador**. Em cada rodada da validação cruzada ele refaz o `fit` de todas as etapas **apenas com a parte de treino daquela rodada**. O vazamento fica impossível por construção.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEMENTE)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

print("Treino:", X_treino.shape, " Teste:", X_teste.shape)

### Tarefa 3 — Montando o Pipeline
1. Crie `modelo_pca`: um `Pipeline` com três etapas:
   - `("padronizar", StandardScaler())`
   - `("pca", PCA(n_components=7))`
   - `("clf", LogisticRegression(max_iter=5000))`
2. Crie `modelo_base`: o mesmo, **sem** a etapa de PCA.
3. Avalie os dois com `cross_val_score(modelo, X_treino, y_treino, cv=cv, scoring="accuracy")` e mostre a **média ± desvio padrão**.

In [ ]:
# Tarefa 3 — seu código aqui
modelo_pca = ...
modelo_base = ...

---
# D — Quantas componentes? E a PCA compensa?

Em vez de escolher *k* "no olho", vamos **testar** vários valores e medir.

### Tarefa 4 — Acurácia × número de componentes
1. Para *k* de **1 a 30**, ajuste `modelo_pca` com `PCA(n_components=k)` e guarde a média e o desvio da acurácia na validação cruzada.
   Dica: `modelo_pca.set_params(pca__n_components=k)`.
2. Faça um gráfico da acurácia média × *k*, com uma **linha horizontal** marcando o `modelo_base` (sem PCA).
3. Qual *k* deu a maior acurácia? Ela é **muito** diferente do modelo sem PCA?

In [ ]:
# Tarefa 4 — seu código aqui
ks = range(1, 31)

**Sua resposta:** _escreva aqui_

### Tarefa 5 — Deixando a variância escolher *k*
Em vez de um número, `PCA(n_components=0.95)` mantém **quantas componentes forem necessárias para guardar 95% da variância**.

1. Ajuste `modelo_pca` com `pca__n_components=0.95` em **todo o treino** (`.fit`).
2. Descubra quantas componentes foram escolhidas: `modelo_pca.named_steps["pca"].n_components_`.
3. Avalie esse modelo na validação cruzada.

In [ ]:
# Tarefa 5 — seu código aqui

### Tarefa 6 (desafio) — E com Random Forest?
Árvores de decisão não sofrem com colinearidade como os modelos lineares. Repita a Tarefa 4 trocando o classificador por `RandomForestClassifier(n_estimators=200, random_state=SEMENTE, n_jobs=-1)` e use apenas `ks = [1, 2, 3, 5, 7, 10, 15, 20, 30]` (para não demorar). A PCA ajuda ou atrapalha a floresta?

In [ ]:
# Tarefa 6 — seu código aqui

**Sua resposta:** _escreva aqui_

---
# E — Avaliação final no conjunto de teste

Só agora, **uma única vez**, usamos o teste.

### Tarefa 7 — O veredito
1. Escolha um modelo (por exemplo, a regressão logística com o `k_melhor` da Tarefa 4).
2. Ajuste-o em **todo o treino**, preveja o **teste** e mostre:
   - a acurácia;
   - a matriz de confusão (rótulos `["maligno", "benigno"]`);
   - o `classification_report`.
3. Quantos tumores **malignos** foram classificados como benignos? Por que esse erro importa mais que o contrário?

In [ ]:
# Tarefa 7 — seu código aqui

**Sua resposta:** _escreva aqui_

---
# Quando a PCA compensa em um modelo?

| Compensa | Não compensa |
|---|---|
| Muitas variáveis **muito correlacionadas** em modelos lineares | Modelos de árvore (Random Forest, Gradient Boosting) |
| Mais variáveis que observações (*p* ≫ *n*): genômica, sensores, texto | Quando é preciso **explicar** o modelo variável por variável |
| Reduzir custo de treino e armazenamento | Poucas variáveis, pouco correlacionadas |
| Remover ruído de medições | Quando a informação que separa as classes está em direções de **pouca** variância |

**Dois lembretes:**
- A PCA **não usa o `y`**: a direção de maior variância não é necessariamente a que melhor separa as classes.
- Num relatório para o médico, "o raio do núcleo é o fator mais importante" é útil; "a PC3 é importante" não é. **Interpretabilidade tem valor.**

## Fique Conectado

- [![YouTube](https://img.icons8.com/ios-filled/40/000000/youtube-play.png)](https://www.youtube.com/@LigaDataScience/videos)  
  Explore nossos vídeos educacionais e webinars sobre ciência de dados, machine learning e inteligência artificial. Inscreva-se para não perder nenhuma atualização!

- [![LinkedIn](https://img.icons8.com/ios-filled/40/000000/linkedin.png)](https://www.linkedin.com/company/liga-data-science-ufpr/)  
  Siga-nos no LinkedIn para as últimas novidades, oportunidades de carreira e networking profissional no campo da ciência de dados.

- [![Instagram](https://img.icons8.com/ios-filled/40/000000/instagram-new.png)](https://www.instagram.com/ligadatascience/)  
  Confira nosso Instagram para conteúdos dos bastidores, destaques de eventos e o dia a dia da Liga Data Science. Faça parte da nossa jornada!

_______

## Autor

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>

## Referências

* <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html">Scikit-learn: PCA</a>
* <a href="https://scikit-learn.org/stable/modules/compose.html">Scikit-learn: Pipelines</a>
* <a href="https://scikit-learn.org/stable/common_pitfalls.html">Scikit-learn: Common pitfalls (data leakage)</a>
* <a href="https://scikit-learn.org/stable/modules/cross_validation.html">Scikit-learn: Cross-validation</a>

## Log de modificações

| Data | Versão | Modificado por | Descrição |
| ----------------- | ------- | ---------- | ---------------------------------- |
| 24-10-2021       | 1.0     | Eduardo Pecora | Criação (aula única de PCA) |
| 09-11-2026       | 2.0     | Claude (a pedido de Eduardo Pécora) | Notebook próprio: vazamento, Pipeline, validação cruzada e escolha de k; cabeçalho/rodapé LDS |

<hr>

## <h3 align="center"> (c) LDS / UFPR 2026. All rights reserved. <h3/>